# 03b — Finetuning DistilGPT2 on Real Legal Datasets (Atticus Contracts / Pile of Law)

**Day 1 — AI Foundations · Practical 3b of 6 · Companion to the "Finetuning & KV Cache" deck**

> **Running in Google Colab:** this notebook uses a real Hugging Face dataset. Training on a
> **T4 GPU** is recommended (Runtime → Change runtime type → GPU). CPU works too, but will be
> noticeably slower. Expected training time on T4: **~2–5 minutes**.

---

## How This Notebook Differs from Notebook 03

Notebook 03 finetuned DistilGPT2 on **16 hand-written contract sentences** — a tiny, self-contained
demo. This notebook replaces those sentences with **real legal datasets** from Hugging Face:

- **`pile-of-law/pile-of-law` (`atticus_contracts` config)** — real contract text from the
  Atticus Project, the same data behind the CUAD (Contract Understanding Atticus Dataset) benchmark
- Optionally, the **full Pile of Law** split — a much broader collection of US legal text

Everything else — the model, tokenizer, `Trainer`, and evaluation — stays the same.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Load a real Hugging Face legal dataset and prepare it for causal language model finetuning
2. Run a complete finetuning loop on **DistilGPT2** using real contract text
3. Generate text **before vs. after** finetuning and observe the shift toward authentic contract
   language (stronger than the 16-sentence demo)
4. Understand the trade-offs between dataset size, training time, and output quality

## Why This Matters for a Law Firm

The 16-sentence demo proves the *concept* — finetuning changes style. This notebook proves
the *practice*: loading a real, curated contract corpus and finetuning on it is exactly what
a production legal-domain model pipeline looks like (just at smaller scale). The Atticus
Contracts dataset is the same data used to train contract-analysis models in academic
research and legal-tech startups.

## Notebook Workflow

```mermaid
flowchart TD
    A["Load atticus_contracts\nfrom Hugging Face"] --> B["Take subset\n(200 examples)"]
    B --> C["Tokenize with\nDistilGPT2 tokenizer"]
    C --> D["Causal LM objective:\npredict next token"]
    D --> E["Trainer.train()\n(3 epochs)"]
    E --> F["Finetuned model\ncheckpoint"]

    G["Prompt:\n'This Agreement shall be governed by...'"] --> H["Generate with\nBASE model"]
    G --> I["Generate with\nFINETUNED model"]
    H --> J["Compare outputs\nside by side"]
    I --> J
```

## Section 1 — Setup

We use **DistilGPT2**, a distilled/compressed version of GPT-2 (see the deck for how knowledge
distillation works). It's small enough to finetune quickly while still showing a clear,
visible stylistic shift after training.

This time we also import `load_dataset` from the `datasets` library — the standard way to
pull real datasets from the Hugging Face Hub.

> **⚠️ Version note:** `pile-of-law` uses a legacy dataset loader script (`pile-of-law.py`).
> The latest `datasets` library (3.x+) dropped support for these scripts, so we pin
> `datasets < 3.0` below. Once `pile-of-law` is converted to Parquet, this pin will no
> longer be needed.

> **Note on scale:** we train on a 200-example subset for ~2–5 minutes on a T4 GPU.
> Production-grade legal-domain finetuning would use the full dataset (10K+ examples) and
> run for longer, but the principle and code are identical.

In [ ]:
# Install dependencies.
# Running in Google Colab: this cell installs everything needed -- just run it.
#
# IMPORTANT: datasets < 3.0 is pinned because pile-of-law still uses a legacy dataset
# loading script, which newer versions of the `datasets` library no longer support.
# This is a known issue — downgrading to the last 2.x release is the recommended fix.
%pip install -q transformers torch "datasets<3.0"

import torch
from transformers import (
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from datasets import load_dataset, Dataset

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

## Section 2 — Load Real Legal Datasets

Instead of hand-writing 16 sentences, we pull real contract text from Hugging Face.

We'll load **two** datasets so you can see what's available:

| Dataset | Config | Content | Approx. Size |
|---------|--------|---------|------------|
| `pile-of-law/pile-of-law` | `atticus_contracts` | Real contract clauses from the Atticus Project (CUAD) | ~10K+ examples |
| `pile-of-law/pile-of-law` | *(default split)* | Broad US legal text (court opinions, admin decisions, etc.) | ~200K+ examples |

We'll **train on `atticus_contracts`** — it's the most directly relevant to contract-drafting
assistance. The full Pile of Law split is shown below as a commented-out alternative; you can
swap to it by changing one line.

We take a **subset of 200 examples** to keep training fast. Increase or decrease this number
to trade off between quality and training time.

In [ ]:
# --- Load atticus_contracts (contract-specific text) ---
# This is our TRAINING dataset.
print("Loading atticus_contracts from pile-of-law...")
atticus_raw = load_dataset("pile-of-law/pile-of-law", "atticus_contracts", split="train")
print(f"Full atticus_contracts size: {len(atticus_raw):,} examples")
print(f"Columns: {atticus_raw.column_names}")

# Take a subset for quick training.
# Increase SUBSET_SIZE for better results (at the cost of longer training).
SUBSET_SIZE = 200
train_dataset = atticus_raw.shuffle(seed=42).select(range(SUBSET_SIZE))
print(f"Using subset: {len(train_dataset)} examples for training\n")

# Show a few examples so you can see what the data looks like
print("Sample atticus_contracts entries:")
for i in range(3):
    text = train_dataset[i]["text"]
    # Truncate very long entries for display
    display_text = text[:200] + "..." if len(text) > 200 else text
    print(f"\n  [{i}] {display_text}")

# --- Load the full Pile of Law split (for comparison only) ---
# Uncomment the lines below and comment out atticus_contracts above to use this instead.
# pile_raw = load_dataset("pile-of-law/pile-of-law", split="train")
# train_dataset = pile_raw.shuffle(seed=42).select(range(SUBSET_SIZE))
# print(f"Full Pile of Law size: {len(pile_raw):,} examples")

## Section 3 — Generate BEFORE Finetuning (Baseline)

Before we touch any weights, let's see how the **untouched base model** completes a
contract-style prompt. This is our baseline — we'll compare against it after finetuning.

In [ ]:
MODEL_NAME = "distilgpt2"

tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default; reuse EOS

base_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
base_model.eval()

prompt = "This Agreement shall be governed by"

def generate_completion(model, prompt, max_new_tokens=40):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.8,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("BEFORE finetuning:\n")
print(generate_completion(base_model, prompt))

Run the cell above a couple of times — with sampling enabled (`do_sample=True`), you'll likely
see the base model drift into generic, non-legal prose. That's expected: it has no particular
bias toward contract-boilerplate continuations yet.

## Section 4 — Tokenize the Dataset

We tokenize the loaded Hugging Face dataset with the same tokenizer the model will train with.
The text column in `atticus_contracts` is `"text"` — we verify this below. `truncation=True`
guards against any unexpectedly long example exceeding the model's context window.

In [ ]:
# Verify which column holds the text (should be 'text' for atticus_contracts)
text_column = "text"
print(f"Using column: '{text_column}'")

def tokenize_fn(examples):
    return tokenizer(examples[text_column], truncation=True, max_length=128)

tokenized_dataset = train_dataset.map(tokenize_fn, batched=True, remove_columns=[text_column])

print(f"\nTokenized dataset: {tokenized_dataset}")
print(f"\nExample tokenized row:")
print(tokenized_dataset[0])

## Section 5 — Finetune with the Causal Language Modeling Objective

`DataCollatorForLanguageModeling(mlm=False)` sets us up for **causal** (next-token-prediction)
language modeling — the same objective used to pretrain GPT-2 itself, just continued on our
contract dataset.

Key difference from Notebook 03: we use **3 epochs** instead of 15, because we have 200 real
examples rather than 16 hand-written ones. More data means fewer epochs needed.

---

### ⏱️ Expected Training Time (Colab T4 GPU)

| Subset Size | Epochs | Approx. Time |
|------------|--------|-------------|
| 200 | 3 | **~2–5 minutes** |
| 500 | 3 | ~5–10 minutes |
| 200 | 5 | ~4–8 minutes |

> Tip: increase `SUBSET_SIZE` in Section 2 and/or `num_train_epochs` below for stronger
> results — but expect proportionally longer training.

In [ ]:
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./distilgpt2-legal-finetuned",
    per_device_train_batch_size=4,
    num_train_epochs=3,           # 200 examples × 3 epochs is enough for a visible shift
    learning_rate=5e-5,
    logging_steps=10,
    save_strategy="no",           # skip checkpoint saving for this quick demo
    report_to=[],                 # disable wandb/tensorboard logging integrations
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

train_result = trainer.train()
print("\nFinal training loss:", train_result.training_loss)

## Section 6 — Generate AFTER Finetuning

Same prompt, same decoding settings, only the model weights have changed. Compare this output
directly against Section 3's baseline.

In [ ]:
model.eval()

print("AFTER finetuning:\n")
print(generate_completion(model, prompt))

## Section 7 — Side-by-Side Comparison

Let's generate several completions from both models on different contract-style prompts,
printed side by side, to get a clearer sense of the shift than a single sample can show.

In [ ]:
test_prompts = [
    "This Agreement shall be governed by",
    "Either party may terminate",
    "The Receiving Party shall",
    "In the event of a breach,",
    "All notices under this Agreement shall",
]

for p in test_prompts:
    print("=" * 90)
    print(f"PROMPT: {p!r}\n")
    print("BASE MODEL:     ", generate_completion(base_model, p, max_new_tokens=40))
    print("FINETUNED MODEL:", generate_completion(model, p, max_new_tokens=40))
    print()

## Section 8 — Key Takeaways

1. **Real data > hand-written data.** Finetuning on 200 real contract examples from
   `atticus_contracts` produces a noticeably stronger and more authentic stylistic shift
   than the 16-sentence demo in Notebook 03. The model absorbs actual contract rhythm, not
   just boilerplate templates.
2. **Finetuning doesn't change the architecture** — every layer, every attention head is
   exactly the same. Only the learned *weights* shift.
3. **The `atticus_contracts` subset is production-grade data.** This is the same dataset
   behind the CUAD benchmark used in legal AI research. Swapping to your firm's own
   precedent bank follows the exact same pattern — just point `load_dataset` at your data.
4. **Training time scales linearly.** 200 examples × 3 epochs ≈ 2–5 min on T4. Double the
   examples → double the time. This is **full finetuning** — every weight updated. Day 2's
   LoRA/QLoRA content covers parameter-efficient alternatives that achieve similar results
   with far less compute.
5. **You can swap datasets with one line.** Change `atticus_contracts` to the full Pile of
   Law split (uncomment the lines in Section 2) to finetune on a broader legal corpus instead.
6. **A note on real deployment:** legal-drafting assistance built this way still requires human
   attorney review — finetuning teaches *style*, not legal correctness, and hallucinated or
   incorrect clause language is a real risk that must be checked by a qualified lawyer.

**Next up:** the *KV Cache Speed Benchmark* notebook — making generation like this fast enough
to use interactively on much longer documents.